# How a single delayed sample moves a closed-loop pole

The controller samples a first-order plant and holds its command until the next sample. Compare immediate application with the same calculation arriving exactly one sample late. Predict which gain produces oscillation before running the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tau,period,gain=.50,.10,4.5    # seconds, seconds, dimensionless proportional gain
samples=50
assert tau>0 and period>0 and gain>=0 and 10<=samples<=200
a=np.exp(-period/tau)
b=1-a

## Derive the homogeneous sampled recurrences
For regulation to zero the direct controller gives x[k+1] = (a - bK)x[k]. With delay the coefficient multiplies x[k-1] instead. Initialise the remembered reading to zero, so the first delayed command is zero.

In [ ]:
def run(k,delay):
    old,x=0.,1.;values=[x]
    for _ in range(samples):
        old,x=x,a*x-b*k*(old if delay else x)
        values.append(x)
    return np.array(values)
poles=np.roots([1,-a,b*gain])
direct=run(gain,False);delayed=run(gain,True)
print(f"a = {a:.4f}, direct pole = {a-b*gain:.4f}, delayed poles = {poles}")
print(f"Upper positive gain limits: immediate {(1+a)/b:.3f}, delayed {1/b:.3f}")
assert np.isclose(direct[1],a-b*gain) and np.isclose(delayed[1],a)
assert np.isclose(np.prod(poles),b*gain)

## Locate the damping and stability limits
The two controllers use identical numerical coefficients and the same plant. Compare the absolute value of the largest pole with the sequence of sampled displacements.

In [ ]:
gains=np.linspace(0,1.15*(1+a)/b,260)
spectral=[np.max(np.abs(np.roots([1,-a,b*k]))) for k in gains]
fig,ax=plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
times=np.arange(samples+1)*period
ax[0].plot(times,direct,".-",ms=3,label="Immediate")
ax[0].plot(times,delayed,".--",ms=3,label="One sample late")
ax[0].set(xlabel="Time (s)",ylabel="Sampled deviation",title="Response from the same initial state")
ax[1].plot(gains,np.abs(a-b*gains),label="Immediate")
ax[1].plot(gains,spectral,"--",label="One sample late")
ax[1].axhline(1,color="0.5",ls=":",label="Unit circle")
ax[1].set(xlabel="Positive gain K",ylabel="Maximum pole modulus",title="Discrete stability margin")
for p in ax:p.grid(alpha=.22);p.legend(fontsize=8)
plt.show()

Set the gain between the two limits and compare the traces. Reduce the sample period by half while leaving the physical time constant unchanged; recompute the gain limits instead of simply reusing the original values. What would change if the delay were fixed in seconds rather than in samples?